# nb_05c — Reflection-equivariant read-out, by construction (E2)

**E1** established the informative null: the frozen S-JEPA does *not* encode the signed left–right axis as a decodable, sign-flipping quantity (learned $\approx$ floor; mirror slope $\approx-0.70$). **E2** asks the constructive question:

> If we *impose* reflection-equivariance on the **read-out**, do we recover exact antisymmetry — and does the axis become more decodable from the *same* frozen tokens?

> **Responsible-use notice.** All results in this notebook are **transductive** — the
> S-JEPA encoder was trained on the very sequences being evaluated — so they carry
> *internal validity only* and make **no** claim of generalization to new sources or
> people. The **source video** (not the clip, not the individual) is the independent unit
> of analysis. The dataset's condition folders (`normal`, `parkinsons`, `stroke`,
> `myopathic`, `cerebralpalsy`) are **dataset annotations, not diagnoses**. The dataset's
> official distribution provides annotations and public video URLs, not raw video; this
> analysis uses derived pose sequences, infers no identity, and redistributes no raw or
> identity-bearing frames. **No institutional ethics determination or completed data-use
> review is yet on record; both must be resolved before submission.**

## 1. Frame averaging over $G=\{I,M\}$

Let $A(x)$ be the per-pair laterality feature of the frozen encoder. Average over the order-2 group (Puny et al., 2022):

$$ \Phi(x) = A(x) - A(Mx)\quad(\text{antisymmetric}),\qquad \Psi(x) = A(x) + A(Mx)\quad(\text{symmetric}). $$

Because $M^2=I$, $\;\Phi(Mx) = A(Mx)-A(x) = -\Phi(x)$ **exactly**, and $\Psi(Mx)=+\Psi(x)$ **exactly**. A *linear* read-out $w^\top\Phi$ therefore satisfies $w^\top\Phi(Mx) = -\,w^\top\Phi(x)$: **mirror slope $=-1$ for *any* weights $w$** — the antisymmetry is a property of the construction, not of training (`../neurips-laterality/docs/figures/fig4_construction.svg`).

The target $y$ is itself exactly antisymmetric, so a **symmetric** read-out on $\Psi$ *cannot* track its sign — a clean falsification control.

**Isolating the cause.** To show any gain comes from the *constraint* and not from capacity or the extra mirror pass, we add two learnable heads on the frozen tokens:
- `eq_mlp`: $s(x)=m(A)-m(A_{\mathrm{mir}})$ with a **shared** MLP $m$ → exactly antisymmetric for any $m$;
- `free_mlp`: an **untied** MLP on $[A;A_{\mathrm{mir}}]$ with $\ge$ the same capacity, seeing both passes but free to ignore the symmetry.

In [1]:
import json, os, subprocess, sys, textwrap
from pathlib import Path

def find_experiment_dir(start=None):
    candidates = []
    if os.getenv("ALEXPOSE_ROOT"):
        env_root = Path(os.environ["ALEXPOSE_ROOT"]).expanduser().resolve()
        candidates.extend([env_root, env_root / "experiments" / "sjepa" / "gavd5-drift"])
    start = Path(start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        candidates.extend([base, base / "experiments" / "sjepa" / "gavd5-drift"])
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "work" / "experiments").is_dir():
            return candidate
    raise FileNotFoundError(f"Cannot locate gavd5-drift from {start}; set ALEXPOSE_ROOT.")


EXPERIMENT_DIR = find_experiment_dir()
NOTEBOOK_DIR = EXPERIMENT_DIR / "neurips-brain-body"
PY = EXPERIMENT_DIR / ".venv" / "bin" / "python"
PY = str(PY if PY.exists() else sys.executable)   # fall back to the running kernel
ART = EXPERIMENT_DIR / "work" / "artifacts" / "real"

def run_experiment(script_relpath):
    """Run a validated standalone experiment script; it regenerates its JSON artifact."""
    script = EXPERIMENT_DIR / script_relpath
    print(f"running {script.name} with {PY} ...")
    proc = subprocess.run([PY, str(script)], cwd=str(EXPERIMENT_DIR),
                          capture_output=True, text=True)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print("STDERR (tail):\n", proc.stderr[-4000:])
        raise RuntimeError(f"{script.name} exited {proc.returncode}")
    return proc

def approx(a, b, tol):
    return abs(float(a) - float(b)) <= tol


In [2]:
# Re-run E2 end-to-end (regenerates idea9_equivariant_readout_result.json).
run_experiment('work/experiments/e2_equivariant_readout.py')
res = json.loads((ART / 'idea9_equivariant_readout_result.json').read_text())
prim = res['primary_cohort']
print('fingerprint', res['fingerprint'][:12], '| primary',
      prim['n_sequences'], 'seq /', prim['n_sources'], 'sources')

running e2_equivariant_readout.py with /Users/pmui/dev/alexpose/experiments/sjepa/gavd5-drift/.venv/bin/python ...


checkpoint 7d13841aceac config={'frames': 64, 'joints': 33, 'coordinate_dim': 3, 'segment_length': 4, 'embed_dim': 96, 'encoder_depth': 4, 'predictor_depth': 2, 'heads': 4}  train_ids=626
642 availability: 642 seq / 94 sources
626 modeled     : 626 seq / 93 sources

=== PRIMARY: 626 modeled cohort ===
{
  "lanes": {
    "A_free": {
      "r2": 0.26839426283742185,
      "mae": 2.037501857331948,
      "sign_consistency": 0.5483870967741935
    },
    "Phi_learned": {
      "r2": 0.31407621628307536,
      "mae": 2.0087864718954767,
      "sign_consistency": 0.5698924731182796
    },
    "Phi_floor": {
      "r2": 0.12079805884860373,
      "mae": 2.1273193413185787,
      "sign_consistency": 0.6666666666666666
    },
    "Psi_learned": {
      "r2": 0.014856559268894176,
      "mae": 2.326829716096456,
      "sign_consistency": 0.5698924731182796
    },
    "B_raw": {
      "r2": 0.9999999999968954,
      "mae": 3.545913241042281e-06,
      "sign_consistency": 1.0
    }
  },
  "mirror_

In [3]:
# ---- Table 2: read-out lanes on 626 primary (repeated-CV R2 + mirror slope)
lanes, slopes, ci = prim['lanes'], prim['mirror_slopes'], prim['repeated_cv_ci95']
def band(k):
    if k in ci:
        d = ci[k]; return f"{d['mean']:.3f} [{d['ci95_lo']:.3f}, {d['ci95_hi']:.3f}]"
    return f"{lanes[k]['r2']:.3f}"
print('E2 read-out — 626 primary')
print(f"  A  — free ridge            R2={band('A_free'):24s} slope={slopes['A_free']:+.4f}")
print(f"  Phi— frame-avg, learned    R2={band('Phi_learned'):24s} slope={slopes['Phi_learned']:+.7f}")
print(f"  Phi— frame-avg, untrained  R2={band('Phi_floor'):24s} slope={slopes['Phi_floor']:+.7f}")
print(f"  Psi— symmetric part        R2={lanes['Psi_learned']['r2']:+.3f}                    slope={slopes['Psi_learned']:+.4f}")
print(f"  B  — raw ceiling           R2={lanes['B_raw']['r2']:.3f}")
sc = ci['Phi_learned_sign_consistency']
print(f"\n  Phi_learned sign consistency {sc['mean']:.3f} [{sc['ci95_lo']:.3f}, {sc['ci95_hi']:.3f}]")

E2 read-out — 626 primary
  A  — free ridge            R2=0.198 [0.175, 0.221]     slope=-0.7035
  Phi— frame-avg, learned    R2=0.273 [0.253, 0.294]     slope=-1.0000001
  Phi— frame-avg, untrained  R2=0.219 [0.175, 0.264]     slope=-1.0000000
  Psi— symmetric part        R2=+0.015                    slope=+1.0000
  B  — raw ceiling           R2=1.000

  Phi_learned sign consistency 0.568 [0.553, 0.582]


In [4]:
# ---- Learnable-head controls: the gain is the CONSTRAINT, not capacity/extra pass
h = prim['learnable_heads']
print(f"  eq_mlp   (shared m: s=m(A)-m(A_mir))  R2={h['eq_mlp']['r2']:+.3f}  slope={h['eq_mlp']['mirror_slope']:+.7f}")
print(f"  free_mlp (untied, >=capacity)         R2={h['free_mlp']['r2']:+.3f}  slope={h['free_mlp']['mirror_slope']:+.3f}")

  eq_mlp   (shared m: s=m(A)-m(A_mir))  R2=+0.243  slope=-1.0000000
  free_mlp (untied, >=capacity)         R2=+0.047  slope=-0.430


In [5]:
# ---- Assert the paper's E2 numbers trace to this freshly-written artifact
Phi = ci['Phi_learned']; Afree = ci['A_free']; Pfloor = ci['Phi_floor']
assert approx(Phi['mean'], 0.273, 0.01), Phi
assert approx(Afree['mean'], 0.198, 0.01), Afree
assert approx(slopes['Phi_learned'], -1.0, 1e-4), slopes['Phi_learned']
assert approx(slopes['Psi_learned'], +1.0, 1e-4), slopes['Psi_learned']
assert approx(lanes['Psi_learned']['r2'], 0.015, 0.01), lanes['Psi_learned']['r2']
assert approx(h['free_mlp']['r2'], 0.047, 0.02), h['free_mlp']
# (1) built-in beats free with DISJOINT stability intervals (partition-stability heuristic)
assert Phi['ci95_lo'] > Afree['ci95_hi'], (Phi, Afree)
# (2) learned edges the untrained floor but intervals OVERLAP -> learning benefit suggestive
assert Phi['mean'] > Pfloor['mean'] and Phi['ci95_lo'] < Pfloor['ci95_hi'], (Phi, Pfloor)
print('OK — E2 numbers match the paper; built-in beats free (disjoint intervals);')
print('     the learning-vs-geometry gap is honestly suggestive (overlapping intervals).')

OK — E2 numbers match the paper; built-in beats free (disjoint intervals);
     the learning-vs-geometry gap is honestly suggestive (overlapping intervals).


## 2. Reading the result

Two observations stand out. **(1) Built-in beats free**, with *disjoint* stability intervals ($\Phi\approx0.273$ vs $A\approx0.198$; a partition-stability heuristic, not a significance test): imposing the geometry unlocks decodable structure the free read-out does not reach from the *same* frozen encoder ($\Phi$ additionally evaluates the mirrored pass $A(Mx)$). **(2) The controls are consistent with the antisymmetric constraint — not capacity or the extra pass — as the source of the gain (they do not isolate a causal decomposition).** The symmetric $\Psi$ cannot predict an antisymmetric target ($\approx0.015$, slope $+1$); the shared-map `eq_mlp` is exactly antisymmetric and a *linear* $\Phi$ is already optimal (nonlinearity adds nothing); the untied `free_mlp` — $\ge$ the same capacity, both passes — collapses to $\approx0.047$ with slope $\approx-0.43$.

**Honesty.** $\Phi_{\text{learned}}\approx0.273$ only *edges* $\Phi_{\text{floor}}\approx0.219$ and their stability intervals **overlap**, so the *learning* benefit is suggestive, not decisive — frame-averaging a random encoder already reaches most of the absolute score — and $\Phi$ stays far below the raw ceiling (sign consistency $\approx0.57 < 0.75$). The recovered axis is real but partial, which motivates pushing symmetry into the **encoder** (`nb_05d`). See `../neurips-laterality/docs/figures/fig5_builtin_beats_emergent.svg`.